This is a notebook for making a very basic implementation of an early exit LLM with some risk control. Mostly to test understanding of core concepts and how to apply them.

Steps:
1. Set up some working open-source LLM. 
2. Implement some measure of risk on the LLM (e.g. diff between full vs layer output). Compute it at each layer for a specific prompt. 
3. Implement early-exit when the risk exceeds some value lambda (choose one randomly/arbitrarily for now).
4. Find some (any) dataset to serve as "validation" (doesn't need to actually be a good dataset). Implement "grid search" for empirically finding a "good" lambda.

Code references:
- https://github.com/aangelopoulos/ltt - LTT
- https://github.com/metodj/rc-eenn - Metod’s paper (early exit NNs for risk control)
- https://github.com/aangelopoulos/conformal-risk/tree/main - conformal risk control paper code
- https://github.com/thomaspzollo/prompt_risk - prompt risk control paper code
- https://github.com/google-research/t5x/tree/main/t5x/contrib/calm - CALM (early exit LLM paper)
- https://colab.research.google.com/github/pytorch/pytorch.github.io/blob/master/assets/hub/huggingface_pytorch-transformers.ipynb - PyTorch transformers (language model example code)

In [2]:
# Imports
import torch
import transformers
from detoxify import Detoxify
from transformers import AutoTokenizer

/opt/anaconda3/envs/llm-risk-control/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load some open source pretrained language models
gpt2 = torch.hub.load('huggingface/transformers', 'modelForCausalLM', 'gpt2')
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
# Maybe try the T5 model (same as CALM paper)
# Some more pretrained models to work with, plus sample code: 
# https://colab.research.google.com/github/pytorch/pytorch.github.io/blob/master/assets/hub/huggingface_pytorch-transformers.ipynb#scrollTo=096dbd52

Using cache found in /Users/andreawynn/.cache/torch/hub/huggingface_transformers_main
/opt/anaconda3/envs/llm-risk-control/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
# Compute measure of risk as e.g. toxicity score for now (later I'll measure difference between intermediate vs final output)
Detoxify('original').predict('hello world')['toxicity']

0.0009625151

In [38]:
# Compute the risk at each layer for a specific prompt. 
gpt2_layers = [module for module in gpt2.modules() if isinstance(module, transformers.models.gpt2.modeling_gpt2.GPT2Block)]
for l in gpt2_layers:
    print(type(l))
print(len(gpt2_layers))

<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
<class 'transformers.models.gpt2.modeling_gpt2.GPT2Block'>
12


In [44]:
text = "Hello, my name is Andrea, what's yours?"
indexed_tokens = tokenizer.encode(text, add_special_tokens=True)

# TODO - try loading from HuggingFace instead? Make sure I can access all individual layers. 

AttributeError: 